In [1]:
#pyg or py data, dataset, dataloader?
# dictionary or Data as output of PROTACDataset?
    #What works with the data loader?
    #What works with pyg funcitons such as num_node_features?



# import torch dataset and dataloader

#from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os
# Import from_smiles from pytorch geometric
from torch_geometric.utils import from_smiles
from torch_geometric.data import Data, Dataset, InMemoryDataset
from torch_geometric.loader import DataLoader
from all_functions import get_node_labels
from rdkit import Chem


    

class ProtacDataset(InMemoryDataset):  #Changed from pytorch Dataset to PyG InMemoryDataset

    def __init__(self, protac_df, transform=None):
        self.protac_df = protac_df
        self.protac_smiles = protac_df['PROTAC SMILES'].tolist()
        self.poi_smiles = protac_df['POI SMILES'].tolist()
        self.e3_smiles = protac_df['E3 SMILES'].tolist()
        self.substructures = ['.'.join(protac_df[['POI SMILES', 'LINKER SMILES','E3 SMILES']].iloc[i].tolist()) for i in range(len(protac_df))] #remove later when I have updated functions to not use split_sort. Possible as the substructures will be already separated in the dataframe

        #node_boundaries_list = [get_node_labels(smiles, substructures_joined) for smiles, substructures_joined in zip(self.protac_smiles, self.substructures)]
        self.node_boundaries = [get_node_labels(smiles, substructures_joined) for smiles, substructures_joined in zip(self.protac_smiles, self.substructures)] #torch.FloatTensor(node_boundaries_list[0])
                                #torch.FloatTensor(dtype=torch.float32)
        
        

    def __len__(self):
        return len(self.protac_df)
    
    def __getitem__(self, idx):
       
        #elem = { 'pyg_data': from_smiles(self.protac_smiles[idx]), # (1024,)}

        #elem = {
        #    'x': x ,            
        #    'edge_index': edge_index,
        #    'edge_attr': edge_attr,
        #    'smiles': smiles,
        #    'node_boundaries': self.node_boundaries[idx], # List of boundary classes for each node # (num_nodes,) # boundary_ligand_nodes
        #}
        
        #x, edge_index, edge_features, smiles = from_smiles(self.protac_smiles[idx])
        #elem = Data(
        #        x=x,  #Node features
        #        edge_index=edge_index,
        #        edge_attr=edge_features,
        #        smiles=smiles,
        #        node_boundaries=self.node_boundaries[idx]) 

        elem = from_smiles(self.protac_smiles[idx])
        elem["x"]=elem["x"].type(torch.float32)
        elem["edge_index"]=elem["edge_index"].type(torch.int64)
        elem["edge_attr"]=elem["edge_attr"].type(torch.float32)
        
        elem["node_boundaries"] = self.node_boundaries[idx]


        return elem


/home/knkn308/.conda/envs/env-protac-toolkit/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from torch_geometric.nn import GraphConv
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


#optimizer = optim.Adam(model.parameters(), lr=0.001)

#GCNConv worked well before, with a test accuracy of around 0.93
#GraphConv: maybe converges faster?
class PROTACSplitter(torch.nn.Module):
    def __init__(self, node_feature_dim, edge_feature_dim):
        super(PROTACSplitter, self).__init__()
        self.conv1 = GraphConv(node_feature_dim, 16)# edge_dim=edge_feature_dim)
        self.linear_layer = torch.nn.Linear(16, 3)  # Output layer for 3 classes
    
    #def forward(self, data_batch):
    def forward(self, node_attr, edge_index):
        #data = batch['pyg_data']
        #print(f"data_batch.x: {data_batch.x}")
        #print(f"data_batch.edge_index: {data_batch.edge_index}")
        #z = self.conv1(data_batch.x, data_batch.edge_index)#, edge_attr)
        z = self.conv1(node_attr, edge_index)#, edge_attr)
        z = F.relu(z)
        y = self.linear_layer(z)
        return y
        
    def train_model(self, training_data, optimizer=optim.Adam, lr = 0.001, batch_size=32, criterion=nn.CrossEntropyLoss(), shuffle=True):
        self.train()
        #training_pyg_data = training_data['pyg_data']
        train_loader = DataLoader(training_data, batch_size=batch_size, shuffle=shuffle)
        total_loss = 0
        #print(train_loader)
        for train_data in train_loader:
            optimizer=optimizer(self.parameters(), lr=lr)
            optimizer.zero_grad()
            node_attr=train_data.x
            edge_index = train_data.edge_index
            #print(type(node_attr))
            #print(node_attr.dtype)
            #print(edge_index.dtype)
            raw_boundary_prediction = self.forward(node_attr, edge_index)             # "RuntimeError: mat1 and mat2 must have the same dtype"
            node_class_targets = train_data['node_boundaries']
            print(node_class_targets.size())
            print(type(node_class_targets))
            print(len(node_class_targets))
            print(raw_boundary_prediction.size())
            print(type(raw_boundary_prediction))
            print(len(raw_boundary_prediction))
            #print(node_class_targets)

            loss = criterion(raw_boundary_prediction, node_class_targets) 
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / batch_size

        return avg_loss

def train_model_v2(model, loader, optimizer, criterion, epoch):
    model.train()
    total_loss = 0
    for data in loader:
        
        optimizer.zero_grad()

        #print(data.edge_index)
        
        out = model(node_attr=data.x, edge_index=data.edge_index)#, edge_attr=data.edge_attr)
        
        
        #num_smiles = len(data.smiles)
        target_boundary_list = []
        for smile_idx, smile in enumerate(data.smiles):
                
            substructure_smiles = data.substructure_smiles[smile_idx]
            protac_smile = data.smiles[smile_idx]
            one_hot_boundary_nodes = one_hot_encode_boundary_nodes(protac_smiles=protac_smile, substructure_smiles=substructure_smiles)
            #print(f'one_hot_boundary_nodes: {one_hot_boundary_nodes}')
            #print(one_hot_boundary_nodes)
            # Convert one-hot encoded vectors to class indices
            target_boundary_for_idx = one_hot_boundary_nodes.argmax(dim=1) #0: POI boundary node, 1: non-boundary node (Linker and ligands), 2: E3 boundary node
            target_boundary_list.append(target_boundary_for_idx)
        
        target = target_boundary_list[0]
        if len(target_boundary_list) > 1:
            for i in range(1, len(target_boundary_list)):
                target = torch.cat((target, target_boundary_list[i]))

        loss = criterion(out, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [3]:
protac_pub_trainset_df = pd.read_csv('../../data/augmented/protac_pub_testset_testing.csv')
protac_pub_testset_df = pd.read_csv('../../data/augmented/protac_pub_trainset_testing.csv')
train_set_pub = ProtacDataset(protac_df=protac_pub_trainset_df)

In [4]:
print(train_set_pub[0])
from torch_geometric.loader import DataLoader
train_loader = DataLoader(train_set_pub, batch_size=32, shuffle=True)
x = next(iter(train_loader))
print(x.node_boundaries)

Data(x=[63, 9], edge_index=[2, 134], edge_attr=[134, 3], smiles='CCC(NC(=O)C1CC(C(=O)CCCCCCCCCCN2CCC3(CC2)CC(C)N(c2ccc(C#N)c(Cl)c2)C3)CN1C(=O)C(NC(=O)C(C)NC)C(C)(C)C)c1ccccc1', node_boundaries=[63])
tensor([0, 0, 0,  ..., 0, 0, 0])


In [5]:
#print(train_set_pub[0])
#train_set_pub[0].node_boundaries.dtype

In [6]:
len(train_set_pub[0].node_boundaries) #0 8, 1 7

print(x.node_boundaries[62+7])

#for i in range(20):
#    print(x.node_boundaries[i])


tensor(0)


In [7]:
#node_feature_dim = train_set_pub[0]['x'][1].num_node_features
node_feature_dim = train_set_pub.num_node_features
edge_feature_dim = train_set_pub.num_edge_features 
model = PROTACSplitter(node_feature_dim, edge_feature_dim)

#optimizer = optim.Adam(model.parameters(), lr=0.001)
#criterion = torch.nn.CrossEntropyLoss() # Note: CrossEntropyLoss in PyTorch expects raw scores (logits) and class indices, not one-hot encoded labels If your labels are one-hot encoded, you might need to convert them to class indices
#trainloader = DataLoader(train_set_pub, batch_size=1, shuffle=True)


In [8]:
#train_model_v2(model, trainloader, optimizer, criterion, epoch=0)

In [9]:
model.train_model(training_data=train_set_pub, batch_size=32)

torch.Size([1539])
<class 'torch.Tensor'>
1539
torch.Size([1539, 3])
<class 'torch.Tensor'>
1539


0.1321219801902771

In [ ]:
x.node_boundaries[0]


In [ ]:


model = PROTACSplitter()
for batch in DataLoader(train_set_pub, batch_size=32):
    # batch['protac_smiles'] = (batch_size, 1024)
    y_hat = model(batch)
    loss = loss_fn(y_hat, batch['node_boundaries'])
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    # Metric
    y_hat = torch.argmax(y_hat, dim=1) # (batch_size, num_nodes) -> (batch_size, 1)
    acc = (y_hat == batch['node_boundaries']).sum() / len(y_hat)
